In [17]:
import pandas as pd
import pickle
import numpy as np

In [18]:
# 1. Load Model
with open('../../models/model_progress.pickle', 'rb') as f:
    data_model = pickle.load(f)

models = data_model['models_dict']
encoders = data_model['encoders']
feature_order = data_model['features']

In [ ]:
# --- FUNGSI PREDIKSI ---
def prediksi_progres_lengkap(user_data, minggu_ke):
    """
    Fungsi ini memprediksi kondisi fisik dan kebutuhan nutrisi pengguna
    untuk minggu tertentu menggunakan kumpulan model Machine Learning.

    ---------------------------------------------------------------------------
    Parameters (Input):
    1. user_data (dict):
        Dictionary berisi data profil awal pengguna. Keys wajib meliputi:
        - 'Age', 'Gender', 'Height_cm', 'Initial_Weight_kg'
        - 'Goal' (Muscle Gain/Weight Loss), 'level' (Beginner/etc.)
        - 'Body_Fat_Percentage_x', 'Workout_Frequency', 'Average_Duration_Minutes'
        - Flag Olahraga: 'Badminton', 'Football', dll (0/1).

    2. minggu_ke (int):
        Minggu keberapa yang ingin diprediksi (Contoh: 1, 4, atau 12).
        Input ini menjadi variabel waktu utama bagi model untuk melihat tren perubahan.

    ---------------------------------------------------------------------------
    Logika:
    
    A. FEATURE ENGINEERING (Data Turunan):
    Menghitung BMI awal dan menentukan Kategori BMI (Underweight/Normal/Obese)
    secara otomatis berdasarkan tinggi dan berat badan input, sehingga user tidak
    perlu menghitung manual.

    B. DATA ENCODING (Konversi Input):
    Mengubah data teks (Gender, Goal, Level) menjadi angka numerik menggunakan 
    LabelEncoder yang sudah dilatih sebelumnya, agar bisa diproses oleh model AI.

    C. MULTI-MODEL PREDICTION (Prediksi Paralel):
    Fungsi ini melakukan looping terhadap dictionary 'models'. 
    Setiap aspek (Berat Badan, Kalori, Protein, Body Fat) memiliki model khusus 
    sendiri-sendiri yang diprediksi secara terpisah namun dalam satu aliran data.

    D. DECODING OUTPUT (Penerjemahan Hasil):
    - Jika output model adalah Angka (misal: Berat Badan), hasil langsung disimpan.
    - Jika output model adalah Kategori (misal: Status BMI), angka hasil prediksi
        dikembalikan menjadi teks (inverse transform) agar mudah dibaca manusia.

    ---------------------------------------------------------------------------
    Output:
    - Dictionary berisi hasil prediksi lengkap untuk minggu tersebut:
    {
        'Weight_kg': 60.5,
        'BMI': 21.0,
        'Daily_Calories': 2500,
        'Target_Protein_g': 150,
        ...
    }
    """


    # A. Hitung Data Turunan & B. Encoding (Sama seperti sebelumnya)
    tinggi_m = user_data['Height_cm'] / 100
    bmi_awal = round(user_data['Initial_Weight_kg'] / (tinggi_m ** 2), 2)
    
    # Hitung kategori BMI awal kasar
    if bmi_awal < 18.5: cat_bmi = 'Underweight'
    elif bmi_awal < 25: cat_bmi = 'Normal'
    elif bmi_awal < 30: cat_bmi = 'Overweight'
    else: cat_bmi = 'Obese'
    
    gender_code = encoders['Gender'].transform([user_data['Gender']])[0]
    goal_code = encoders['Goal'].transform([user_data['Goal']])[0]
    level_code = encoders['level'].transform([user_data['level']])[0]
    bmi_cat_code = encoders['BMI_Category_x'].transform([cat_bmi])[0]

    # C. Susun Array Input
    input_row = [
        user_data['Age'], 
        gender_code, 
        user_data['Height_cm'], 
        user_data['Initial_Weight_kg'],
        bmi_awal, 
        bmi_cat_code, 
        user_data['Body_Fat_Category'], 
        user_data['Body_Fat_Percentage_x'],
        goal_code, 
        user_data['Workout_Frequency'], 
        user_data['Average_Duration_Minutes'], 
        level_code,
        user_data['Badminton'], 
        user_data['Football'], 
        user_data['Basketball'],
        user_data['Volleyball'], 
        user_data['Swim'],
        minggu_ke
    ]
    
    # Bungkus jadi DataFrame
    input_df = pd.DataFrame([input_row], columns=feature_order)
    
    # D. Lakukan Prediksi UNTUK SEMUA MODEL
    hasil = {}
    
    # Kita loop semua model yang ada di dictionary 'models'
    # target_name = nama kolom (misal: Weight_kg, Daily_Calories, Progress_Status_Encoded)
    # info = isinya {'model': ..., 'type': ...}
    for target_name, info in models.items():
        
        # Prediksi nilainya
        nilai_prediksi = info['model'].predict(input_df)[0]
        
        # Cek tipe model (Angka atau Kategori?)
        if info['type'] == 'categorical':
            # Kalau kategori, kita harus terjemahkan balik (Decode)
            # Nama encoder biasanya nama target tanpa "_Encoded"
            nama_asli = target_name.replace('_Encoded', '')
            
            # Terjemahkan angka ke teks
            teks = encoders[nama_asli].inverse_transform([int(nilai_prediksi)])[0]
            hasil[nama_asli] = teks
        else:
            # Kalau angka, langsung simpan
            hasil[target_name] = nilai_prediksi
            
    return hasil


In [20]:
input_user = {
    'Age': 25, 
    'Gender': 'Male', 
    'Height_cm': 175, 
    'Initial_Weight_kg': 60,
    'Body_Fat_Category': 2, 
    'Body_Fat_Percentage_x': 15.0,
    'Goal': 'Muscle Gain', 
    'Workout_Frequency': 4, 
    'Average_Duration_Minutes': 60,
    'level': 'Beginner',
    'Badminton': 0, 
    'Football': 1, 
    'Basketball': 0, 
    'Volleyball': 0, 
    'Swim': 0
}

print(f"User: Pria, 25th, 60kg -> Goal: Muscle Gain")

for minggu in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    output = prediksi_progres_lengkap(input_user, minggu_ke=minggu)
    
    print(f"\n{'='*10} MINGGU KE-{minggu} {'='*10}")
    
    # Kelompokkan output biar enak dibaca
    print(f"[FISIK]")
    print(f"  Berat Badan     : {output['Weight_kg']:.2f} kg")
    print(f"  BMI             : {output['BMI']:.2f} ({output['BMI_Category_y']})")
    print(f"  Body Fat        : {output['Body_Fat_Percentage_y']:.1f}%")
    
    print(f"[NUTRISI HARIAN]")
    print(f"  Kalori          : {output['Daily_Calories']:.0f} kkal")
    print(f"  Air Minum       : {output['Daily_Water_ml']:.0f} ml")
    print(f"  Gula (Limit)    : {output['Limit_Sugar_g']:.1f} g")
    
    print(f"[MAKRO NUTRISI]")
    print(f"  Protein         : {output['Target_Protein_g']:.1f} g")
    print(f"  Karbo           : {output['Target_Carbs_g']:.1f} g")
    print(f"  Lemak           : {output['Target_Fat_g']:.1f} g")
    print(f"  Serat           : {output['Target_Fiber_g']:.1f} g")
    print(f"  Kalsium         : {output['Target_Calcium_mg']:.0f} mg")
    print(f"  Kolesterol (Max): {output['Limit_Cholesterol_mg']:.0f} mg")

User: Pria, 25th, 60kg -> Goal: Muscle Gain

========== MINGGU KE-1 ==========
[FISIK]
  Berat Badan     : 60.23 kg
  BMI             : 19.46 (Normal)
  Body Fat        : 14.9%
[NUTRISI HARIAN]
  Kalori          : 2703 kkal
  Air Minum       : 2741 ml
  Gula (Limit)    : 67.0 g
[MAKRO NUTRISI]
  Protein         : 204.3 g
  Karbo           : 326.8 g
  Lemak           : 58.4 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-2 ==========
[FISIK]
  Berat Badan     : 60.26 kg
  BMI             : 19.51 (Normal)
  Body Fat        : 14.8%
[NUTRISI HARIAN]
  Kalori          : 2703 kkal
  Air Minum       : 2741 ml
  Gula (Limit)    : 67.0 g
[MAKRO NUTRISI]
  Protein         : 204.3 g
  Karbo           : 326.8 g
  Lemak           : 58.4 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-3 ==========
[FISIK]
  Berat Badan     : 60.30 kg
  BMI             : 19.54 (Normal)
  Body Fat       


========== MINGGU KE-4 ==========
[FISIK]
  Berat Badan     : 60.42 kg
  BMI             : 19.58 (Normal)
  Body Fat        : 14.6%
[NUTRISI HARIAN]
  Kalori          : 2706 kkal
  Air Minum       : 2752 ml
  Gula (Limit)    : 67.1 g
[MAKRO NUTRISI]
  Protein         : 204.5 g
  Karbo           : 327.1 g
  Lemak           : 58.5 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-5 ==========
[FISIK]
  Berat Badan     : 60.54 kg
  BMI             : 19.61 (Normal)
  Body Fat        : 14.6%
[NUTRISI HARIAN]
  Kalori          : 2706 kkal
  Air Minum       : 2752 ml
  Gula (Limit)    : 67.1 g
[MAKRO NUTRISI]
  Protein         : 204.5 g
  Karbo           : 327.1 g
  Lemak           : 58.5 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-6 ==========
[FISIK]
  Berat Badan     : 60.62 kg
  BMI             : 19.65 (Normal)
  Body Fat        : 14.5%
[NUTRISI HARIAN]
  Kalori          